# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [11]:
!pip install pandas

In [14]:

pip install --upgrade pandas


  Using cached pandas-2.3.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (91 kB)
Using cached pandas-2.3.3-cp313-cp313-macosx_11_0_arm64.whl (10.7 MB)
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install --upgrade numpy pandas


Note: you may need to restart the kernel to use updated packages.


In [4]:
#Definition of the tables.
import pandas as pd

data = {
    'table': ['account', 'client', 'loan'],
    'definition': [
        'account_id district_id frequency date',
        'client_id birth_number district_id ',
        'loan_id account_id date amount duration payments status'] }
df = pd.DataFrame(data)
print(df)



     table                                         definition
0  account              account_id district_id frequency date
1   client                client_id birth_number district_id 
2     loan  loan_id account_id date amount duration paymen...


In [5]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [6]:
print(text_tables)

account: account_id district_id frequency date
client: client_id birth_number district_id 
loan: loan_id account_id date amount duration payments status


In [7]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [9]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question = "What are the different frequencies of account updates available in each district?")

In [10]:
print(return_OAI(pqt1))

{
    "tables": ["account", "district"]
}


In [11]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question= "Which clients have taken loans greater than 5000 and what is the status of those loans?")

In [12]:
print(return_OAI(pqt3))

{
    "tables": ["client", "loan"]
}


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [13]:
#Creating the prompt, with the user questions and the tables definitions.
pqt4 = prompt_question_tables.format(tables=text_tables, question = "How many loans have been issued per district?")
print(return_OAI(pqt4))

{
    "tables": ["loan", "account"]
}


In [14]:
pqt5 = prompt_question_tables.format(tables=text_tables, question = "List all clients along with their account frequencies and loan statuses.")
print(return_OAI(pqt5))

```json
{
    "tables": ["client", "account", "loan"]
}
```


Short one-page report (summary)

Report Summary:

Objective:
Determine which database tables are necessary to answer a user query about clients, account frequencies, and loan statuses using GPT-5-mini.

Process:

Constructed a DataFrame representing tables (account, client, loan) with column definitions.

Converted the DataFrame into a textual summary for GPT.

Formulated a prompt instructing GPT to return table names in JSON format relevant to the user’s question.

Sent the prompt via return_OAI() function to GPT.

Observations:

GPT correctly identified that the client, account, and loan tables are relevant.

Minor hallucinations may occur if column definitions are ambiguous or formatted as free text.

Using a structured list of columns instead of a space-separated string improves accuracy.

Challenges / Variations:

GPT sometimes suggests non-existent tables if definitions are unclear.

Typos in the prompt (e.g., “Questyion”) can confuse model reasoning.

Providing column-level detail reduces hallucinations.

Lessons Learned:

Clean and structured table definitions improve LLM output accuracy.

Always check prompt formatting; small errors can propagate in LLM reasoning.

Converting DataFrame to text summary is convenient but may lose structured detail.

A simple pandas DataFrame is an effective way to generate a prompt for GPT querying.